# Exploratory Data Analysis - International Football Results & FIFA Rankings

In this notebook, we perform an exploratory data analysis (EDA) on our preprocessed international football match results merged with historical FIFA rankings. 

The goal is to understand the distributions of goals, historical trends, confederation-level patterns, and the correlation between FIFA rankings and match outcomes.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Set project path relative to workspace
os.chdir("..")
print("Current directory:", os.getcwd())

## 1. Load Preprocessed Datasets

In [ ]:
matches = pd.read_csv("data/processed/matches_cleaned.csv")
rankings = pd.read_csv("data/processed/rankings_cleaned.csv")

print(f"Matches dataset: {matches.shape[0]} rows, {matches.shape[1]} columns")
print(f"Rankings dataset: {rankings.shape[0]} rows, {rankings.shape[1]} columns")
matches['date'] = pd.to_datetime(matches['date'])
matches.head()

## 2. Outcome Distribution (Home Win vs. Draw vs. Away Win)

Let's see the proportion of match outcomes from the perspective of the home team. This helps us understand the extent of home advantage in international football.

In [ ]:
outcome_counts = matches['outcome'].value_counts(normalize=True) * 100
print(outcome_counts)

fig = px.pie(
    names=outcome_counts.index,
    values=outcome_counts.values,
    title="Distribution of International Match Outcomes (Home Team Perspective)",
    labels={'names': 'Outcome', 'values': 'Percentage (%)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 3. Distribution of Goals (Home vs. Away)

Let's visualize how many goals are typically scored per match by home and away teams. This is essential for verifying if goalscoring distributions approximate a Poisson distribution.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=matches['home_score'], name='Home Goals', opacity=0.75, marker_color='#1f77b4'))
fig.add_trace(go.Histogram(x=matches['away_score'], name='Away Goals', opacity=0.75, marker_color='#ff7f0e'))

fig.update_layout(
    title_text="Distribution of Goals Scored (Home vs. Away)",
    xaxis_title_text="Goals Scored",
    yaxis_title_text="Match Count",
    barmode='overlay',
    template="plotly_white"
)
fig.show()

## 4. Top 15 Scoring Nations

Which international teams have scored the most goals in matches since 1993?

In [ ]:
# Calculate total goals scored by each team (both home and away)
home_goals = matches.groupby('home_team')['home_score'].sum()
away_goals = matches.groupby('away_team')['away_score'].sum()
total_goals = home_goals.add(away_goals, fill_value=0).sort_values(ascending=False).head(15)

fig = px.bar(
    x=total_goals.values,
    y=total_goals.index,
    orientation='h',
    title="Top 15 Scoring International Teams (Since 1993)",
    labels={'x': 'Total Goals Scored', 'y': 'Team'},
    color=total_goals.values,
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template="plotly_white")
fig.show()

## 5. FIFA Rank vs. Match Outcomes

Does a higher FIFA ranking correlate with winning matches? Let's calculate the ranking difference (Home Rank - Away Rank) and check how it relates to goal differences.

In [ ]:
# Difference in ranks: lower rank is better (e.g. 1st is better than 10th)
# Negative difference means home team is better ranked
matches['rank_diff'] = matches['home_rank'] - matches['away_rank']

fig = px.box(
    matches,
    x='outcome',
    y='rank_diff',
    title="FIFA Rank Difference by Match Outcome",
    labels={'outcome': 'Outcome (Home Team Perspective)', 'rank_diff': 'Home Rank - Away Rank'},
    color='outcome',
    color_discrete_map={'W': '#2ca02c', 'D': '#9467bd', 'L': '#d62728'}
)
fig.update_layout(template="plotly_white")
fig.show()

## 6. Average Goals by Confederation

Let's see if confederations exhibit differences in average goals scored per match (Home + Away goals).

In [ ]:
matches['total_goals'] = matches['home_score'] + matches['away_score']
# Drop matches where confederation is missing
conf_goals = matches.dropna(subset=['home_confederation']).groupby('home_confederation')['total_goals'].mean().sort_values()

fig = px.bar(
    x=conf_goals.values,
    y=conf_goals.index,
    orientation='h',
    title="Average Goals per Match by Home Team Confederation",
    labels={'x': 'Average Goals', 'y': 'Confederation'},
    color=conf_goals.values,
    color_continuous_scale='Magma'
)
fig.update_layout(template="plotly_white")
fig.show()

## 7. Correlation Heatmap

Let's look at the correlation between numeric features of interest.

In [ ]:
corr_cols = ['home_score', 'away_score', 'goal_difference', 'total_goals', 'home_rank', 'away_rank', 'rank_diff', 'neutral']
corr_matrix = matches[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Match Features')
plt.show()